In [1]:
import requests
from bs4 import BeautifulSoup

url = "https://books.toscrape.com/"

response = requests.get(url)
response.encoding = "utf-8"

print(response.status_code)

200


##### 200 means roughly: Server received my request and successfully returned the page.

In [2]:
soup = BeautifulSoup(response.text, "html.parser")

books = soup.find_all("article", class_="product_pod")

print(len(books))

20


In [3]:
print(books[0])

<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>


###### The important concept: Website ->Request -> HTML ->BeautifulSoup -> Find repeating HTML element (article.product_pod) -> 20 individual products ->Extract:Title,Price,Rating,Availability,URL ->Pandas DataFrame -> CSV/Excel/SQL

##### Our goal is to turn that messy HTML into clean data. Use '"<a> to get the book title."

In [4]:
book = books[0]

title = book.h3.a["title"]

print(title)

A Light in the Attic


##### Extract the price

In [5]:
price = book.find("p", class_="price_color").text

print(price)

£51.77


##### Extract Availability

In [6]:
availability = book.find("p", class_="instock").text.strip()

print(availability)

In stock


###### Extract Rating

In [7]:
rating = book.find("p", class_="star-rating")["class"]

print(rating)

['star-rating', 'Three']


In [8]:
rating = book.find("p", class_="star-rating")["class"][1]

print(rating)

Three


###### Put it all together

In [9]:
book = books[0]

title = book.h3.a["title"]
price = book.find("p", class_="price_color").text
rating = book.find("p", class_="star-rating")["class"][1]
availability = book.find("p", class_="instock").text.strip()

print(title)
print(price)
print(rating)
print(availability)

A Light in the Attic
£51.77
Three
In stock


###### There is currency, we do not want that, we want it as a float.

In [10]:
price = book.find("p", class_="price_color").text
price = float(price.replace("£", ""))

print(price)

51.77


##### Do a loop through the 20 books in the page.

In [11]:
titles = []
prices = []
ratings = []
availability_list = []

for book in books:
    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    price = float(price.replace("£", ""))
    rating = book.find("p", class_="star-rating")["class"][1]
    availability = book.find("p", class_="instock").text.strip()

    titles.append(title)
    prices.append(price)
    ratings.append(rating)
    availability_list.append(availability)

it looks like this
book 1 → extract values → add to lists
book 2 → extract values → add to lists
book 3 → extract values → add to lists
...
book 20 → extract values → add to lists

In [12]:
print(len(titles))
print(titles[:5])
print(prices[:5])

20
['A Light in the Attic', 'Tipping the Velvet', 'Soumission', 'Sharp Objects', 'Sapiens: A Brief History of Humankind']
[51.77, 53.74, 50.1, 47.82, 54.23]


###### Turn everything into a Pandas DataFrame

In [13]:
import pandas as pd

df = pd.DataFrame({
    "title": titles,
    "price": prices,
    "rating": ratings,
    "availability": availability_list
})

df.head()

,title,price,rating,availability
0,A Light in the Attic,51.77,Three,In stock
1,Tipping the Velvet,53.74,One,In stock
2,Soumission,50.10,One,In stock
3,Sharp Objects,47.82,Four,In stock
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   title         20 non-null     object 
 1   price         20 non-null     float64
 2   rating        20 non-null     object 
 3   availability  20 non-null     object 
dtypes: float64(1), object(3)
memory usage: 768.0+ bytes


##### URL Structure https://books.toscrape.com/catalogue/page-2.html

So there is a predictable pattern. 

In [15]:
for page in range(1, 51):
    print(page)

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50


In [16]:
for page in range(1, 51):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    print(url)

https://books.toscrape.com/catalogue/page-1.html
https://books.toscrape.com/catalogue/page-2.html
https://books.toscrape.com/catalogue/page-3.html
https://books.toscrape.com/catalogue/page-4.html
https://books.toscrape.com/catalogue/page-5.html
https://books.toscrape.com/catalogue/page-6.html
https://books.toscrape.com/catalogue/page-7.html
https://books.toscrape.com/catalogue/page-8.html
https://books.toscrape.com/catalogue/page-9.html
https://books.toscrape.com/catalogue/page-10.html
https://books.toscrape.com/catalogue/page-11.html
https://books.toscrape.com/catalogue/page-12.html
https://books.toscrape.com/catalogue/page-13.html
https://books.toscrape.com/catalogue/page-14.html
https://books.toscrape.com/catalogue/page-15.html
https://books.toscrape.com/catalogue/page-16.html
https://books.toscrape.com/catalogue/page-17.html
https://books.toscrape.com/catalogue/page-18.html
https://books.toscrape.com/catalogue/page-19.html
https://books.toscrape.com/catalogue/page-20.html
https://b

Create empty list first

In [17]:
all_titles = []
all_prices = []
all_ratings = []
all_availability = []

In [18]:
for page in range(1, 3):

    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url)
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")

    books = soup.find_all("article", class_="product_pod")

    print("Page:", page, "Books:", len(books))

Page: 1 Books: 20
Page: 2 Books: 20


We do a test for the first 2 pages and get page 1 and 2. Now we have 2 levels, we need to create a loop inside another loop. We use a nested loop.

In [19]:
for book in books:

    title = book.h3.a["title"]
    price = book.find("p", class_="price_color").text
    rating = book.find("p", class_="star-rating")["class"][1]
    availability = book.find("p", class_="instock").text.strip()

    all_titles.append(title)
    all_prices.append(price)
    all_ratings.append(rating)
    all_availability.append(availability)

In [20]:
print(len(all_titles))

20


In [21]:
import pandas as pd

df_all = pd.DataFrame({
    "title": all_titles,
    "price": all_prices,
    "rating": all_ratings,
    "availability": all_availability
})

In [22]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

df_all["rating"] = df_all["rating"].map(rating_map)

In [23]:
df_all["price"] = (
    df_all["price"]
    .str.replace("£", "", regex=False)
    .astype(float)
)

In [24]:
df_all.head()

,title,price,rating,availability
0,In Her Wake,12.84,1,In stock
1,How Music Works,37.32,2,In stock
2,Foolproof Preserving: A Guide to Small Batch J...,30.52,3,In stock
3,Chase Me (Paris Nights #2),25.27,5,In stock
4,Black Dust,34.53,5,In stock


In [25]:
df_all.to_excel("books_scraped.xlsx", index=False)

In [26]:
import os
print(os.getcwd())

C:\Users\justi\OneDrive\Desktop\SMU\Python Self\Mining
